# Medium Articles Ingestion — EdTech Content Data Pipeline

This notebook ingests the Hugging Face dataset `BEE-spoke-data/medium-articles-en`, keeps only articles related to **AI, Data, and Cloud**, performs basic data-quality checks, standardizes the records, and saves the curated output to the project's `data/` folder.

### Azure pipeline context

This notebook handles the **source ingestion and preprocessing** step. The resulting JSON file can then be uploaded to **Azure Blob Storage** as the raw/curated source and copied with **Azure Data Factory (ADF)** into **Azure Data Lake Storage Gen2 (ADLS Gen2)** for downstream processing.

**Flow:** Hugging Face → filter/clean/standardize → JSON → Azure Blob Storage → ADF → ADLS Gen2


## 1. Install required packages

In [ ]:
%pip install datasets pandas

   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ------------------ --------------------- 262.1/559.1 kB ? eta -:--:--
   -------------------------------------- 559.1/559.1 kB 970.3 kB/s eta 0:00:00
   ---------------------------------------- 0.0/798.3 kB ? eta -:--:--
   ---------------------------------------  786.4/798.3 kB 4.6 MB/s eta 0:00:01
   ---------------------------------------- 798.3/798.3 kB 2.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/27.9 MB ? eta -:--:--
   - -------------------------------------- 1.3/27.9 MB 679.5 MB/s eta 0:00:01
   --- ------------------------------------ 2.4/27.9 MB 10.6 MB/s eta 0:00:03
   ---- ----------------------------------- 3.1/27.9 MB 7.1 MB/s eta 0:00:04
   ----- ---------------------------------- 3.9/27.9 MB 6.0 MB/s eta 0:00:04
   ------ --------------------------------- 4.5/27.9 MB 5.7 MB/s eta 0:00:05
   ------- -----


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports

In [ ]:
from datasets import load_dataset
from pathlib import Path
import pandas as pd
import ast
import json

d:\_SDA-WCD-DataEngineeringBootcamp\TechHub-Group3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Configure output path

If this notebook is inside a `notebooks/` folder, this saves to the existing project-level `data/` folder.

In [ ]:
from pathlib import Path

DATA_DIR = Path("../../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = DATA_DIR / "medium_articles.json"

print("Output file:", OUTPUT_FILE.resolve())

Output file: D:\_SDA-WCD-DataEngineeringBootcamp\TechHub-Group3\notebooks\data\medium_articles.json


## 4. Load the Medium dataset

For development, you can optionally use a smaller slice such as `train[:5000]`. For the final run, use the full `train` split.

In [ ]:
# Final run:
dataset = load_dataset(
    "BEE-spoke-data/medium-articles-en",
    split="train"
)

# Faster development alternative:
# dataset = load_dataset(
#     "BEE-spoke-data/medium-articles-en",
#     split="train[:5000]"
# )

print(dataset)
print("\nColumns:")
print(dataset.column_names)

d:\_SDA-WCD-DataEngineeringBootcamp\TechHub-Group3\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sarah\.cache\huggingface\hub\datasets--BEE-spoke-data--medium-articles-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 4509/4509 [00:00<00:00, 6232.01 exam

Dataset({
    features: ['title', 'text', 'url', 'authors', 'timestamp', 'tags', 'token_count'],
    num_rows: 171340
})

Columns:
['title', 'text', 'url', 'authors', 'timestamp', 'tags', 'token_count']


## 5. Inspect sample records

In [ ]:
for i in range(3):
    article = dataset[i]

    print("=" * 80)
    print("TITLE:", article.get("title"))
    print("AUTHORS:", article.get("authors"))
    print("TAGS:", article.get("tags"))
    print("TOKENS:", article.get("token_count"))
    print("URL:", article.get("url"))
    print()

TITLE: Create Barchart in Android Studio
AUTHORS: ['Kartik Ganiga']
TAGS: ['UI', 'Android', 'Charts', 'Development', 'Android App Development']
TOKENS: 707
URL: https://medium.com/@karthikganiga007/create-barchart-in-android-studio-14943339a211

TITLE: 3 Mistakes Developers Make When They’re in a Hurry
AUTHORS: ['Szymon Adamiak']
TAGS: ['Programming', 'Education', 'Software Development', 'Development', 'Learning To Code']
TOKENS: 644
URL: https://medium.com/better-programming/3-mistakes-developers-make-when-theyre-in-a-hurry-29a8a46109dd

TITLE: Hollywood To Go
AUTHORS: ['Tre L. Loadholt']
TAGS: ['Dreams', 'Audio Poems', 'Art', 'Hollywood']
TOKENS: 277
URL: https://medium.com/a-cornered-gurl/hollywood-to-go-7faaabcd81a3



## 6. Define AI, Data, and Cloud keywords

Specific terms are used instead of broad words such as `data`, `cloud`, or `model` to reduce false positives.

In [ ]:
TOPIC_KEYWORDS = {
    "AI": [
        "artificial intelligence",
        "machine learning",
        "deep learning",
        "generative ai",
        "genai",
        "large language model",
        "large language models",
        "llm",
        "llms",
        "natural language processing",
        "nlp",
        "computer vision",
        "neural network",
        "neural networks",
        "transformer",
        "transformers",
        "tensorflow",
        "pytorch",
        "scikit-learn"
    ],

    "Data": [
        "data science",
        "data scientist",
        "data engineering",
        "data engineer",
        "data analytics",
        "data analysis",
        "big data",
        "data pipeline",
        "data pipelines",
        "etl",
        "elt",
        "data warehouse",
        "data warehousing",
        "data lake",
        "data lakes",
        "database",
        "databases",
        "sql",
        "postgresql",
        "spark",
        "apache spark",
        "airflow",
        "apache airflow",
        "dbt",
        "pandas"
    ],

    "Cloud": [
        "cloud computing",
        "cloud architecture",
        "cloud infrastructure",
        "cloud engineering",
        "cloud engineer",
        "aws",
        "amazon web services",
        "azure",
        "microsoft azure",
        "google cloud",
        "google cloud platform",
        "gcp",
        "serverless",
        "kubernetes",
        "docker",
        "cloud native"
    ]
}

## 7. Normalize tags

In [ ]:
def parse_tags(tags):
    if tags is None:
        return []

    if isinstance(tags, list):
        return [str(tag) for tag in tags]

    if isinstance(tags, str):
        if not tags.strip():
            return []

        try:
            parsed = ast.literal_eval(tags)
            if isinstance(parsed, list):
                return [str(tag) for tag in parsed]
        except (ValueError, SyntaxError):
            pass

        return [tags]

    return [str(tags)]


example_tags = dataset[0].get("tags")

print("Original:")
print(example_tags)

print("\nParsed:")
print(parse_tags(example_tags))

Original:
['UI', 'Android', 'Charts', 'Development', 'Android App Development']

Parsed:
['UI', 'Android', 'Charts', 'Development', 'Android App Development']


## 8. Topic classifier

An article may belong to more than one category, for example `AI, Data, Cloud`.

In [ ]:
def classify_topics(title, tags):
    title = (title or "").lower()

    tag_list = parse_tags(tags)
    tags_text = " ".join(tag_list).lower()

    searchable_text = f"{title} {tags_text}"

    matched_topics = []

    for topic, keywords in TOPIC_KEYWORDS.items():
        if any(keyword in searchable_text for keyword in keywords):
            matched_topics.append(topic)

    return matched_topics

## 9. Test the classifier

In [ ]:
for i in range(min(20, len(dataset))):
    article = dataset[i]

    topics = classify_topics(
        article.get("title"),
        article.get("tags")
    )

    if topics:
        print("=" * 80)
        print("TITLE:", article.get("title"))
        print("TAGS:", parse_tags(article.get("tags")))
        print("TOPICS:", topics)

TITLE: A/B Testing Platforms: Build vs Buy
TAGS: ['A B Testing', 'Data Engineering', 'Data Analytics']
TOPICS: ['Data']


## 10. Filter to AI/Data/Cloud articles

In [ ]:
def is_relevant(example):
    topics = classify_topics(
        example.get("title"),
        example.get("tags")
    )
    return len(topics) > 0


filtered_dataset = dataset.filter(is_relevant)

print("Original articles:", len(dataset))
print("Relevant AI/Data/Cloud articles:", len(filtered_dataset))

Filter: 100%|██████████| 171340/171340 [00:12<00:00, 13747.05 examples/s]

Original articles: 171340
Relevant AI/Data/Cloud articles: 17502


## 11. Convert the filtered dataset to pandas

In [ ]:
df = filtered_dataset.to_pandas()

print("Shape:", df.shape)
df.head()

Shape: (17502, 7)


,title,text,url,authors,timestamp,tags,token_count
0,A/B Testing Platforms: Build vs Buy,"However, there are also many problems with exi...",https://medium.com/growth-book/a-b-testing-pla...,['Graham Mcnicoll'],2020-11-23 15:27:55.557000+00:00,"['A B Testing', 'Data Engineering', 'Data Anal...",578
1,Google Cloud Pub/Sub Ordered Delivery,The Google Cloud Pub/Sub team is happy to anno...,https://medium.com/google-cloud/google-cloud-p...,['Kamal Aboul-Hosn'],2020-10-19 14:48:55.184000+00:00,"['Pub Sub', 'Google Cloud Platform', 'Distribu...",4750
2,Solving Business Problems with Analytics: Work...,Recap (You can skip this section to the next h...,https://medium.com/analytics-vidhya/solving-bu...,['Mark Styx'],2020-03-05 17:56:37.038000+00:00,"['Communication', 'Business', 'Analytics Life ...",674
3,Image Captioning with Attention: Part 2,Model Training\n\nIn the first part of the art...,https://medium.com/analytics-vidhya/image-capt...,['Artyom Makarov'],2020-12-15 16:34:45.404000+00:00,"['Deep Learning', 'Image Captioning', 'Compute...",1000
4,Building Ocelot API Gateway Microservices with...,Building Ocelot API Gateway Microservice on .N...,https://medium.com/aspnetrun/building-ocelot-a...,['Mehmet Özkaya'],2021-04-01 15:15:07.392000+00:00,"['Docker', 'Api Gateway', 'Microservices', 'As...",8384


## 12. Add topic classification

In [ ]:
df["topic"] = df.apply(
    lambda row: classify_topics(
        row["title"],
        row["tags"]
    ),
    axis=1
)

df[["title", "tags", "topic"]].head(10)

,title,tags,topic
0,A/B Testing Platforms: Build vs Buy,"['A B Testing', 'Data Engineering', 'Data Anal...",[Data]
1,Google Cloud Pub/Sub Ordered Delivery,"['Pub Sub', 'Google Cloud Platform', 'Distribu...","[Data, Cloud]"
2,Solving Business Problems with Analytics: Work...,"['Communication', 'Business', 'Analytics Life ...",[Data]
3,Image Captioning with Attention: Part 2,"['Deep Learning', 'Image Captioning', 'Compute...",[AI]
4,Building Ocelot API Gateway Microservices with...,"['Docker', 'Api Gateway', 'Microservices', 'As...",[Cloud]
5,We will tackle this together stay with me :).,"['Integration', 'AWS', 'AWS Lambda', 'Aws Cogn...",[Cloud]
6,Latest Updates on Google Data Analytics (Augus...,"['Google Analytics', 'Bigquery', 'Google Data ...",[Data]
7,Building Go App with Gitlab Runner on Azure |P...,"['Gitlab Ci', 'Cloud', 'Golang', 'Gitlab', 'Az...",[Cloud]
8,"Making the Invisible Visible. At Square, the D...","['Engineering', 'Webhooks', 'AWS', 'Square', '...",[Cloud]
9,Poker? Done that. Now the next challenge…,"['AI', 'Poker', 'Algorithms', 'Artificial Inte...",[AI]


## 13. Data-quality checks and cleaning

In [ ]:
print("Before cleaning:", len(df))

# Required fields
df = df.dropna(subset=["title", "text", "url"])

# Remove blank required values
df = df[
    (df["title"].astype(str).str.strip() != "") &
    (df["text"].astype(str).str.strip() != "") &
    (df["url"].astype(str).str.strip() != "")
]

# Remove duplicate URLs
df = df.drop_duplicates(subset=["url"])

# Remove duplicate title/content combinations
df = df.drop_duplicates(subset=["title", "text"])

df = df.reset_index(drop=True)

print("After cleaning:", len(df))

Before cleaning: 17502
After cleaning: 17373


## 14. Normalize author values

In [ ]:
def normalize_authors(authors):
    if authors is None:
        return ""

    if isinstance(authors, list):
        return ", ".join(str(a) for a in authors)

    if isinstance(authors, str):
        if not authors.strip():
            return ""

        try:
            parsed = ast.literal_eval(authors)
            if isinstance(parsed, list):
                return ", ".join(str(a) for a in parsed)
        except (ValueError, SyntaxError):
            pass

        return authors

    return str(authors)

## 15. Standardize to the project's common content schema

In [ ]:
standardized_df = pd.DataFrame({
    "source": ["Medium"] * len(df),

    "category": df["topic"].apply(
        lambda topics: ", ".join(topics)
    ),

    "title": df["title"],

    "author": df["authors"].apply(
        normalize_authors
    ),

    "publication_date": pd.to_datetime(
        df["timestamp"],
        errors="coerce",
        utc=True
    ).dt.strftime("%Y-%m-%d"),

    "description": [""] * len(df),

    "url": df["url"],

    "content": df["text"],

    "tags": df["tags"].apply(
        parse_tags
    )
})

print("Final shape:", standardized_df.shape)
print("\nColumns:")
print(standardized_df.columns.tolist())

standardized_df.head()

Final shape: (17373, 9)

Columns:
['source', 'category', 'title', 'author', 'publication_date', 'description', 'url', 'content', 'tags']


,source,category,title,author,publication_date,description,url,content,tags
0,Medium,Data,A/B Testing Platforms: Build vs Buy,Graham Mcnicoll,2020-11-23 15:27:55.557000+00:00,,https://medium.com/growth-book/a-b-testing-pla...,"However, there are also many problems with exi...","[A B Testing, Data Engineering, Data Analytics]"
1,Medium,"Data, Cloud",Google Cloud Pub/Sub Ordered Delivery,Kamal Aboul-Hosn,2020-10-19 14:48:55.184000+00:00,,https://medium.com/google-cloud/google-cloud-p...,The Google Cloud Pub/Sub team is happy to anno...,"[Pub Sub, Google Cloud Platform, Distributed S..."
2,Medium,Data,Solving Business Problems with Analytics: Work...,Mark Styx,2020-03-05 17:56:37.038000+00:00,,https://medium.com/analytics-vidhya/solving-bu...,Recap (You can skip this section to the next h...,"[Communication, Business, Analytics Life Cycle..."
3,Medium,AI,Image Captioning with Attention: Part 2,Artyom Makarov,2020-12-15 16:34:45.404000+00:00,,https://medium.com/analytics-vidhya/image-capt...,Model Training\n\nIn the first part of the art...,"[Deep Learning, Image Captioning, Computer Vis..."
4,Medium,Cloud,Building Ocelot API Gateway Microservices with...,Mehmet Özkaya,2021-04-01 15:15:07.392000+00:00,,https://medium.com/aspnetrun/building-ocelot-a...,Building Ocelot API Gateway Microservice on .N...,"[Docker, Api Gateway, Microservices, Aspnetcor..."


## 16. Inspect topic distribution

In [ ]:
topic_counts = (
    df["topic"]
    .explode()
    .value_counts()
)

print(topic_counts)

topic
Data     10240
AI        8088
Cloud     3380
Name: count, dtype: int64


## 17. Manually validate a random sample

Review these records to check whether the keyword rules are producing relevant AI/Data/Cloud results.

In [ ]:
sample = standardized_df.sample(
    min(20, len(standardized_df)),
    random_state=42
)

for _, row in sample.iterrows():
    print("=" * 80)
    print("TITLE:", row["title"])
    print("CATEGORY:", row["category"])
    print("TAGS:", row["tags"])
    print("URL:", row["url"])

TITLE: The Jungle of Koalas, Pandas, Optimus and Spark
CATEGORY: AI, Data
TAGS: ['Technology', 'Data Science', 'Business', 'Artificial Intelligence', 'Machine Learning']
URL: https://towardsdatascience.com/the-jungle-of-koalas-pandas-optimus-and-spark-dd486f873aa4
TITLE: Building an Asp.Net Core Windows Service Task Scheduler
CATEGORY: Cloud
TAGS: ['Quartz', 'Task Scheduler', 'Aspnetcore', 'Azure', 'Windows Services']
URL: https://medium.com/better-programming/asp-net-core-windows-service-task-scheduler-daily-weekly-monthly-700a569d502a
TITLE: In Search of the Perfect Music Dataset
CATEGORY: Data
TAGS: ['Data', 'Music', 'Data Science']
URL: https://medium.com/atchai/in-search-of-the-perfect-music-dataset-ed7e111d3b7e
TITLE: Rabbit Plagues, Genetic Algorithms
CATEGORY: Data
TAGS: ['Biology', 'Data Science', 'Computer Science', 'Evolution']
URL: https://mark-s-cleverley.medium.com/rabbit-plagues-genetic-algorithms-d165643514d8
TITLE: Practical Machine Learning and Rails
CATEGORY: AI
TAGS

## 18. Save the curated dataset to JSON

In [ ]:
records = standardized_df.to_dict(orient="records")

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=2,
        default=str
    )

print(f"Saved {len(records)} Medium articles")
print(f"Location: {OUTPUT_FILE.resolve()}")

Saved 17373 Medium articles
Location: D:\_SDA-WCD-DataEngineeringBootcamp\TechHub-Group3\notebooks\data\medium_articles.json


## 19. Verify the saved file

In [ ]:
with open(
    OUTPUT_FILE,
    "r",
    encoding="utf-8"
) as f:
    saved_data = json.load(f)

print("Records saved:", len(saved_data))

if saved_data:
    print("\nFirst record:")
    print(json.dumps(
        saved_data[0],
        indent=2,
        ensure_ascii=False
    ))

Records saved: 17373

First record:
{
  "source": "Medium",
  "category": "Data",
  "title": "A/B Testing Platforms: Build vs Buy",
  "author": "Graham Mcnicoll",
  "publication_date": "2020-11-23 15:27:55.557000+00:00",
  "description": "",
  "url": "https://medium.com/growth-book/a-b-testing-platforms-build-vs-buy-dfb8604e77e",
  "content": "However, there are also many problems with existing 3rd party platforms. Usually they use their own event tracking, meaning that it’s impossible to test against your existing metrics or data warehouse (Growth Book, by the way, does let you do this). It also means yet another place you’re sending your user data, which might lead to you being out of compliance with the various privacy laws. The kinds of metrics you can test against are typically limited too, with most only supporting binomial events (yes/no, or clicked/didn’t click).\n\nIf you open up the platform for many of your team to start tests, you can end up with interfering tests and make 

## Next Azure step

The notebook stops after producing `data/medium_articles.json`.

For the Azure architecture, the next stages can be:

1. Upload the generated JSON file to **Azure Blob Storage**.
2. Use **Azure Data Factory (ADF)** to ingest/copy it.
3. Store the pipeline output in **Azure Data Lake Storage Gen2 (ADLS Gen2)**.
4. Apply the team's transformation/warehouse layer after ingestion.

This keeps the notebook responsible for **source extraction, filtering, cleaning, and standardization**, while Azure handles the cloud ingestion/orchestration portion of the project.
